Import library

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent.parent

BASE_PATH = PROJECT_ROOT / "data" / "processed" / "movies_metadata.csv" 

RAW_MOVIES_PATH = PROJECT_ROOT / "data" / "raw" / "movies_metadata.csv"
PROCESSED_CREDITS_PATH = PROJECT_ROOT / "data" / "processed" / "credits.csv"
RAW_KEYWORDS_PATH = PROJECT_ROOT / "data" / "raw" / "keywords.csv"

OUTPUT_PATH = PROJECT_ROOT / "data" / "features" / "07_rating_features.csv"



In [3]:
#Initializing and loading up dataframes and storing them into variables to use
base_processed_df = pd.read_csv(BASE_PATH)
credits_processed_df = pd.read_csv(PROCESSED_CREDITS_PATH)
movies_raw_df = pd.read_csv(RAW_MOVIES_PATH)
keywords_raw_df = pd.read_csv(RAW_KEYWORDS_PATH)


/var/folders/4j/5hl0kjh1529gl0sgpv5h03_00000gn/T/ipykernel_38239/1721070842.py:4: DtypeWarning: Columns (10: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_raw_df = pd.read_csv(RAW_MOVIES_PATH)


credits_raw_df good for id conversion.
movies_raw not ready
keywords good
convert base


In [4]:
movies_raw_df.dtypes

adult                        str
belongs_to_collection        str
budget                       str
genres                       str
homepage                     str
id                           str
imdb_id                      str
original_language            str
original_title               str
overview                     str
popularity                object
poster_path                  str
production_companies         str
production_countries         str
release_date                 str
revenue                  float64
runtime                  float64
spoken_languages             str
status                       str
tagline                      str
title                        str
video                     object
vote_average             float64
vote_count               float64
dtype: object

base_processed remove duplicates
drop duplicates for credits raw_df
drop  movies_raw dupe
drop duples for keywords raw_df

In [5]:
#Converts ID into integer so it can merge together.
for data in [base_processed_df,movies_raw_df,keywords_raw_df,credits_processed_df]:
    data["id"] = pd.to_numeric(data['id'],errors= 'coerce').astype('Int64')
    print(data.dtypes)

id                Int64
release_date        str
budget              str
revenue         float64
runtime         float64
popularity      float64
dtype: object
adult                        str
belongs_to_collection        str
budget                       str
genres                       str
homepage                     str
id                         Int64
imdb_id                      str
original_language            str
original_title               str
overview                     str
popularity                object
poster_path                  str
production_companies         str
production_countries         str
release_date                 str
revenue                  float64
runtime                  float64
spoken_languages             str
status                       str
tagline                      str
title                        str
video                     object
vote_average             float64
vote_count               float64
dtype: object
id          Int64
keywords      str


In [6]:
#Dropping ID dupes with this for loop
for data in [base_processed_df,movies_raw_df,keywords_raw_df,credits_processed_df]:
    rows_before = len(data)

    data.drop_duplicates(subset='id',keep='first',inplace=True)

    rows_after = len(data)

#checking for id dupe sum which should equal to 0.
for data in [base_processed_df,movies_raw_df,keywords_raw_df,credits_processed_df]:
    print(data['id'].duplicated().sum()) 

0
0
0
0


Filtering out columns below from the raw dataset.

In [7]:
credits_data = credits_processed_df.loc[:, ['id', 'director','cast_size','crew_size']].copy()
base_data = base_processed_df.loc[:, ['id','release_date','budget','revenue','runtime','popularity']].copy()
movies_raw_columns = movies_raw_df.loc[:, ['id','vote_average','genres','vote_count','belongs_to_collection','production_companies','spoken_languages','production_countries','title','original_title','overview','original_language','status' ]].copy()



In [8]:
#Dropping NA values in column id.
for data in [credits_data,base_data,movies_raw_columns,keywords_raw_df]:
    data.dropna(subset='id',inplace=True)

In [9]:

#Merging all datasets together into merged_df using id
merged_df = (
    base_data.merge(movies_raw_columns,on='id',how='left',validate='one_to_one')
    .merge(credits_data,on='id',how='left',validate='one_to_one')
    .merge(keywords_raw_df,on='id',how='left',validate='one_to_one')
)

merged_df.columns #checking if it all merged which it did

Index(['id', 'release_date', 'budget', 'revenue', 'runtime', 'popularity',
       'vote_average', 'genres', 'vote_count', 'belongs_to_collection',
       'production_companies', 'spoken_languages', 'production_countries',
       'title', 'original_title', 'overview', 'original_language', 'status',
       'director', 'cast_size', 'crew_size', 'keywords'],
      dtype='str')

In [10]:
#creating features with release_date and making separate columns based off their year, month, and decade.

merged_df['release_date'] = pd.to_datetime(merged_df['release_date'],errors='coerce')

merged_df['release_year'] = merged_df['release_date'].dt.year.astype("Int64")

merged_df['release_month'] = merged_df['release_date'].dt.month.astype("Int64")

merged_df['release_decade'] = (merged_df['release_year'] // 10) *10

In [11]:
#returns percentile of vote_count and we can decide what value we want to use to drop the rows for vote_count
def compare_vote_cutoffs(data):
    votes = data["vote_count"]

    for percentile in [0.25, 0.50, 0.75]:
        cutoff = votes.quantile(percentile)

        rows_kept = (votes >= cutoff).sum()
        rows_dropped = len(data) - rows_kept

        print(
            f"{percentile:.0%} percentile | "
            f"Minimum votes: {cutoff:.2f} | "
            f"Rows dropped: {rows_dropped:,} | "
            f"Rows kept: {rows_kept:,}"
        )



In [12]:
compare_vote_cutoffs(merged_df)
min_vote_count =34 #fixed amount to drop the amount of rows refer to the function above to drop howwever many rows you want.

filtered_merged_df = merged_df.loc[merged_df["vote_count"] >= min_vote_count].copy()


print(f"Rows dropped: {len(merged_df) - len(filtered_merged_df)}") #Prints rows dropped with rows before and rows after.
print(f"Rows remaining: {len(filtered_merged_df)}")

25% percentile | Minimum votes: 3.00 | Rows dropped: 9,293 | Rows kept: 36,140
50% percentile | Minimum votes: 10.00 | Rows dropped: 22,518 | Rows kept: 22,915
75% percentile | Minimum votes: 34.00 | Rows dropped: 33,925 | Rows kept: 11,508
Rows dropped: 33925
Rows remaining: 11508


In [13]:
filtered_merged_df.columns

Index(['id', 'release_date', 'budget', 'revenue', 'runtime', 'popularity',
       'vote_average', 'genres', 'vote_count', 'belongs_to_collection',
       'production_companies', 'spoken_languages', 'production_countries',
       'title', 'original_title', 'overview', 'original_language', 'status',
       'director', 'cast_size', 'crew_size', 'keywords', 'release_year',
       'release_month', 'release_decade'],
      dtype='str')

In [14]:
#saving file into data/features
OUTPUT_PATH.parent.mkdir(parents=True,exist_ok=True) 
filtered_merged_df.to_csv(OUTPUT_PATH,index=False)
print(f"Saved: {OUTPUT_PATH}")

Saved: /Users/tor/Documents/GitHub/css-dsml-fall2027/data/features/07_rating_features.csv
